### 1. Load Data

In [273]:
import pandas as pd
import plotly.express as px

In [274]:
df = pd.read_csv('../meat_price_history.csv')
df.head()

,Year,Month,Month-number,Data_Item,Value,Units
0,1970,January,1,Pork byproduct value,4.3,Cents per pound of retail equivalent
1,1970,January,1,Pork gross farm value,52.3,Cents per pound of retail equivalent
2,1970,January,1,Pork net farm value,48.0,Cents per pound of retail equivalent
3,1970,January,1,Pork wholesale value,71.2,Cents per pound of retail equivalent
4,1970,January,1,Pork retail value,81.4,Cents per pound of retail equivalent


### 2. Dataset Overview

In [275]:
scm_summary = pd.DataFrame({
    'SCM_Stage': [
        'Stage 1: Farm Production',
        'Stage 1: Farm Production',
        'Stage 1: Farm Production',
        'Stage 2: Wholesale',
        'Stage 3: Retail',
        'Stage 4: Price Spreads',
        'Stage 4: Price Spreads',
        'Stage 4: Price Spreads'
    ],
    'Category': [
        'Gross farm value',
        'Byproduct value',
        'Net farm value',
        'Wholesale value / composite',
        'Retail value / composite',
        'Farm to retail price spread',
        'Farm to wholesale price spread',
        'Wholesale to retail price spread'
    ],
    'Pork': ['✓', '✓', '✓', '✓', '✓', '✓', '✓', '✓'],
    'Choice Beef': ['✓', '✓', '✓', '✓', '✓', '✓', '✓', '✓'],
    'All Fresh Beef': ['—', '—', '—', '—', '✓', '—', '—', '—'],
    'Chicken': ['—', '—', '—', '✓', '✓', '—', '—', '✓']
})

scm_summary

,SCM_Stage,Category,Pork,Choice Beef,All Fresh Beef,Chicken
0,Stage 1: Farm Production,Gross farm value,✓,✓,—,—
1,Stage 1: Farm Production,Byproduct value,✓,✓,—,—
2,Stage 1: Farm Production,Net farm value,✓,✓,—,—
3,Stage 2: Wholesale,Wholesale value / composite,✓,✓,—,✓
4,Stage 3: Retail,Retail value / composite,✓,✓,✓,✓
5,Stage 4: Price Spreads,Farm to retail price spread,✓,✓,—,—
6,Stage 4: Price Spreads,Farm to wholesale price spread,✓,✓,—,—
7,Stage 4: Price Spreads,Wholesale to retail price spread,✓,✓,—,✓


###### This table summarizes how the dataset maps to the meat supply chain. Pork and choice beef include farm, wholesale, retail, and spread variables. Chicken includes wholesale, retail, and wholesale-to-retail spread variables. All fresh beef appears only as a retail-level measure.

### 3. Data Quality Check

In [276]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13080 entries, 0 to 13079
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Year          13080 non-null  int64  
 1   Month         13080 non-null  object 
 2   Month-number  13080 non-null  int64  
 3   Data_Item     13080 non-null  object 
 4   Value         12630 non-null  float64
 5   Units         13080 non-null  object 
dtypes: float64(1), int64(2), object(3)
memory usage: 613.3+ KB


In [277]:
df.isna().sum()

Year              0
Month             0
Month-number      0
Data_Item         0
Value           450
Units             0
dtype: int64

In [278]:
missing_values = df[df['Value'].isna()]

missing_values.sample(10)

,Year,Month,Month-number,Data_Item,Value,Units
2319,1981,February,2,Retail-wholesale spread for broiler composite,NaN,Cents per retail pound
2897,1983,July,7,Wholesale broiler composite,NaN,Cents per retail pound
2437,1981,August,8,Wholesale broiler composite,NaN,Cents per retail pound
3437,1985,October,10,Wholesale broiler composite,NaN,Cents per retail pound
3477,1985,December,12,Wholesale broiler composite,NaN,Cents per retail pound
4439,1989,December,12,Retail-wholesale spread for broiler composite,NaN,Cents per retail pound
339,1971,August,8,All fresh beef retail value,NaN,Cents per pound of retail equivalent
2599,1982,April,4,Retail-wholesale spread for broiler composite,NaN,Cents per retail pound
2876,1983,June,6,All fresh beef retail value,NaN,Cents per pound of retail equivalent
3859,1987,July,7,Retail-wholesale spread for broiler composite,NaN,Cents per retail pound


In [279]:
missing_values['Data_Item'].value_counts()

Data_Item
All fresh beef retail value                      210
Wholesale broiler composite                      120
Retail-wholesale spread for broiler composite    120
Name: count, dtype: int64

In [280]:
df['Data_Item'].unique()

array(['Pork byproduct value', 'Pork gross farm value',
       'Pork net farm value', 'Pork wholesale value', 'Pork retail value',
       'Pork farm to retail price spread',
       'Pork farm to wholesale price spread',
       'Pork Wholesale to retail price spread',
       'Choice beef byproduct value', 'Choice beef gross farm value',
       'Choice beef net farm value', 'Choice beef wholesale value',
       'Choice beef retail value',
       'Choice beef farm to retail price spread',
       'Choice beef farm to wholesale price spread',
       'Choice beef wholesale to retail price spread',
       'All fresh beef retail value', 'Wholesale broiler composite',
       'Retail broiler composite',
       'Retail-wholesale spread for broiler composite'], dtype=object)

In [281]:
df['Units'].unique()

array(['Cents per pound of retail equivalent', 'Cents per retail pound'],
      dtype=object)

### 4. Feature Engineering

In [282]:
# Map each Data_Item to its corresponding Supply Chain Management (SCM) stage
stage_map = {

    # FARM
    'Pork gross farm value': 'Farm',
    'Pork net farm value': 'Farm',
    'Pork byproduct value': 'Farm',

    'Choice beef gross farm value': 'Farm',
    'Choice beef net farm value': 'Farm',
    'Choice beef byproduct value': 'Farm',

    # WHOLESALE
    'Pork wholesale value': 'Wholesale',
    'Choice beef wholesale value': 'Wholesale',
    'Wholesale broiler composite': 'Wholesale',

    # RETAIL
    'Pork retail value': 'Retail',
    'Choice beef retail value': 'Retail',
    'All fresh beef retail value': 'Retail',
    'Retail broiler composite': 'Retail',

    # SPREADS
    'Pork farm to retail price spread': 'Spread',
    'Pork farm to wholesale price spread': 'Spread',
    'Pork Wholesale to retail price spread': 'Spread',

    'Choice beef farm to retail price spread': 'Spread',
    'Choice beef farm to wholesale price spread': 'Spread',
    'Choice beef wholesale to retail price spread': 'Spread',

    'Retail-wholesale spread for broiler composite': 'Spread'
}

In [283]:
# Create a new column 'SCM_Stage' by mapping the 'Data_Item' values using the stage_map
df['SCM_Stage'] = df['Data_Item'].map(stage_map)

In [284]:
# Display the first 20 rows of the 'Data_Item' and 'SCM_Stage' columns to verify the mapping    
df[['Data_Item', 'SCM_Stage']].head(20)

,Data_Item,SCM_Stage
0,Pork byproduct value,Farm
1,Pork gross farm value,Farm
2,Pork net farm value,Farm
3,Pork wholesale value,Wholesale
4,Pork retail value,Retail
5,Pork farm to retail price spread,Spread
6,Pork farm to wholesale price spread,Spread
7,Pork Wholesale to retail price spread,Spread
8,Choice beef byproduct value,Farm
9,Choice beef gross farm value,Farm


In [285]:
# Map each Data_Item to its corresponding Meat Type
meat_map = {

    # PORK
    'Pork byproduct value': 'Pork',
    'Pork gross farm value': 'Pork',
    'Pork net farm value': 'Pork',
    'Pork wholesale value': 'Pork',
    'Pork retail value': 'Pork',
    'Pork farm to retail price spread': 'Pork',
    'Pork farm to wholesale price spread': 'Pork',
    'Pork Wholesale to retail price spread': 'Pork',

    # BEEF
    'Choice beef byproduct value': 'Beef',
    'Choice beef gross farm value': 'Beef',
    'Choice beef net farm value': 'Beef',
    'Choice beef wholesale value': 'Beef',
    'Choice beef retail value': 'Beef',
    'Choice beef farm to retail price spread': 'Beef',
    'Choice beef farm to wholesale price spread': 'Beef',
    'Choice beef wholesale to retail price spread': 'Beef',
    'All fresh beef retail value': 'Beef',

    # CHICKEN
    'Wholesale broiler composite': 'Chicken',
    'Retail broiler composite': 'Chicken',
    'Retail-wholesale spread for broiler composite': 'Chicken'
}

In [286]:
# Create a new column 'Meat_Type' by mapping the 'Data_Item' values using the meat_map
df['Meat_Type'] = df['Data_Item'].map(meat_map)

In [287]:
df[['Data_Item', 'Meat_Type']].sample(20)

,Data_Item,Meat_Type
6871,Choice beef wholesale value,Beef
10026,Pork farm to wholesale price spread,Pork
3461,Pork gross farm value,Pork
9970,Choice beef net farm value,Beef
806,Pork Wholesale to retail price spread,Pork
4008,Choice beef byproduct value,Beef
12217,Wholesale broiler composite,Chicken
4397,Wholesale broiler composite,Chicken
11175,Choice beef wholesale to retail price spread,Beef
5605,Pork farm to retail price spread,Pork


In [288]:
# Map each Data_Item to its corresponding Metric Type (Value, Spread, Byproduct)
metric_map = {

    'Pork byproduct value': 'Byproduct',
    'Choice beef byproduct value': 'Byproduct',

    'Pork gross farm value': 'Value',
    'Pork net farm value': 'Value',
    'Pork wholesale value': 'Value',
    'Pork retail value': 'Value',

    'Choice beef gross farm value': 'Value',
    'Choice beef net farm value': 'Value',
    'Choice beef wholesale value': 'Value',
    'Choice beef retail value': 'Value',

    'Wholesale broiler composite': 'Value',
    'Retail broiler composite': 'Value',

    'Pork farm to retail price spread': 'Spread',
    'Pork farm to wholesale price spread': 'Spread',
    'Pork Wholesale to retail price spread': 'Spread',

    'Choice beef farm to retail price spread': 'Spread',
    'Choice beef farm to wholesale price spread': 'Spread',
    'Choice beef wholesale to retail price spread': 'Spread',

    'Retail-wholesale spread for broiler composite': 'Spread'
}

In [289]:
df['Metric_Type'] = df['Data_Item'].map(metric_map)

In [290]:
retail = df[df['SCM_Stage'] == 'Retail']

In [291]:
df['SCM_Stage'].value_counts()

SCM_Stage
Spread       4584
Farm         4032
Retail       2568
Wholesale    1896
Name: count, dtype: int64

In [292]:
# Check all unit labels by meat type
df.groupby(['Meat_Type', 'Units']).size()

Meat_Type  Units                               
Beef       Cents per pound of retail equivalent    6048
Chicken    Cents per pound of retail equivalent      36
           Cents per retail pound                  1620
Pork       Cents per pound of retail equivalent    5376
dtype: int64

### 5. Data Quality — Unit Inconsistency (2025 Broiler)

In [ ]:
# Create Chicken Inconsistency Dataset
chicken_inconsistency = df[
    (df['Meat_Type'] == 'Chicken') &
    (df['Units'] == 'Cents per pound of retail equivalent')
]

chicken_inconsistency


,Year,Month,Month-number,Data_Item,Value,Units,SCM_Stage,Meat_Type,Metric_Type
12857,2025,January,1,Wholesale broiler composite,110.974608,Cents per pound of retail equivalent,Wholesale,Chicken,Value
12858,2025,January,1,Retail broiler composite,242.689255,Cents per pound of retail equivalent,Retail,Chicken,Value
12859,2025,January,1,Retail-wholesale spread for broiler composite,131.714647,Cents per pound of retail equivalent,Spread,Chicken,Spread
12877,2025,February,2,Wholesale broiler composite,121.651032,Cents per pound of retail equivalent,Wholesale,Chicken,Value
12878,2025,February,2,Retail broiler composite,243.146408,Cents per pound of retail equivalent,Retail,Chicken,Value
12879,2025,February,2,Retail-wholesale spread for broiler composite,121.495376,Cents per pound of retail equivalent,Spread,Chicken,Spread
12897,2025,March,3,Wholesale broiler composite,107.686616,Cents per pound of retail equivalent,Wholesale,Chicken,Value
12898,2025,March,3,Retail broiler composite,245.211812,Cents per pound of retail equivalent,Retail,Chicken,Value
12899,2025,March,3,Retail-wholesale spread for broiler composite,137.525196,Cents per pound of retail equivalent,Spread,Chicken,Spread
12917,2025,April,4,Wholesale broiler composite,117.009032,Cents per pound of retail equivalent,Wholesale,Chicken,Value


In [294]:
chicken_inconsistency[
    ['Year', 'Month', 'Data_Item', 'Value', 'Units']
]

,Year,Month,Data_Item,Value,Units
12857,2025,January,Wholesale broiler composite,110.974608,Cents per pound of retail equivalent
12858,2025,January,Retail broiler composite,242.689255,Cents per pound of retail equivalent
12859,2025,January,Retail-wholesale spread for broiler composite,131.714647,Cents per pound of retail equivalent
12877,2025,February,Wholesale broiler composite,121.651032,Cents per pound of retail equivalent
12878,2025,February,Retail broiler composite,243.146408,Cents per pound of retail equivalent
12879,2025,February,Retail-wholesale spread for broiler composite,121.495376,Cents per pound of retail equivalent
12897,2025,March,Wholesale broiler composite,107.686616,Cents per pound of retail equivalent
12898,2025,March,Retail broiler composite,245.211812,Cents per pound of retail equivalent
12899,2025,March,Retail-wholesale spread for broiler composite,137.525196,Cents per pound of retail equivalent
12917,2025,April,Wholesale broiler composite,117.009032,Cents per pound of retail equivalent


In [295]:
chicken_inconsistency['Year'].value_counts()

Year
2025    36
Name: count, dtype: int64

In [296]:
df.loc[
    (df['Meat_Type'] == 'Chicken') &
    (df['Units'] == 'Cents per pound of retail equivalent'),
    'Units'
] = 'Cents per retail pound'

In [297]:
df.groupby(['Meat_Type', 'Units']).size()

Meat_Type  Units                               
Beef       Cents per pound of retail equivalent    6048
Chicken    Cents per retail pound                  1656
Pork       Cents per pound of retail equivalent    5376
dtype: int64

### 6. Visualizations

In [298]:
# create a new dataframe with only rows where 'Value' is not null
# not null values are preserved in original df since they were just not introducted in early years, so we can drop the null values for now and work with the clean dataframe
clean_df = df.dropna(subset=['Value']).copy()

# Add Spread_Type classification column
def classify_spread(item):
    item = item.lower()
    if 'farm to retail' in item:
        return 'Farm to Retail'
    elif 'farm to wholesale' in item:
        return 'Farm to Wholesale'
    elif 'wholesale to retail' in item or 'retail-wholesale' in item:
        return 'Wholesale to Retail'
    else:
        return 'Other'

clean_df['Spread_Type'] = clean_df['Data_Item'].apply(classify_spread)

# Create spread subset
spread_df = clean_df[clean_df['Metric_Type'] == 'Spread']

clean_df.head()

,Year,Month,Month-number,Data_Item,Value,Units,SCM_Stage,Meat_Type,Metric_Type,Spread_Type
0,1970,January,1,Pork byproduct value,4.3,Cents per pound of retail equivalent,Farm,Pork,Byproduct,Other
1,1970,January,1,Pork gross farm value,52.3,Cents per pound of retail equivalent,Farm,Pork,Value,Other
2,1970,January,1,Pork net farm value,48.0,Cents per pound of retail equivalent,Farm,Pork,Value,Other
3,1970,January,1,Pork wholesale value,71.2,Cents per pound of retail equivalent,Wholesale,Pork,Value,Other
4,1970,January,1,Pork retail value,81.4,Cents per pound of retail equivalent,Retail,Pork,Value,Other


In [299]:
retail = clean_df[clean_df['SCM_Stage'] == 'Retail']
farm = clean_df[clean_df['SCM_Stage'] == 'Farm']
wholesale = clean_df[clean_df['SCM_Stage'] == 'Wholesale']
spread = clean_df[clean_df['SCM_Stage'] == 'Spread']

In [300]:
fig = px.line(
    retail,
    x='Year',
    y='Value',
    color='Meat_Type',
    title='Retail Meat Prices Over Time'
)

fig.show()

###### This chart shows how retail meat prices changed between 1970 and 2025. Retail prices represent the final consumer-facing stage of the supply chain.

#### Pork Values

In [301]:
# Create Pork Value Dataset
pork_values = clean_df[
    (clean_df['Meat_Type'] == 'Pork') &
    (clean_df['Metric_Type'] == 'Value')
]

In [302]:
pork_values['Data_Item'].unique()

array(['Pork gross farm value', 'Pork net farm value',
       'Pork wholesale value', 'Pork retail value'], dtype=object)

In [303]:
# Remove Extra Pork Metrics, so i see only the farm, wholesale, and retail values for pork price transmission analysis
pork_values = pork_values[
    ~pork_values['Data_Item'].isin([
        'Pork byproduct value',
        'Pork gross farm value'
    ])
]

In [304]:
# Plot the line chart for pork price transmission across the supply chain
fig = px.line(
    pork_values,
    x='Year',
    y='Value',
    color='SCM_Stage',
    title='Pork Price Transmission Across the Supply Chain'
)

fig.show()

###### This chart compares pork prices across farm, wholesale, and retail supply-chain stages. Retail prices remain consistently higher than farm and wholesale values, reflecting processing, transportation, and retail margins.

#### Beef Values

In [305]:
beef_values = clean_df[
    (clean_df['Meat_Type'] == 'Beef') &
    (clean_df['Metric_Type'] == 'Value')
]

In [306]:
beef_values = beef_values[
    ~beef_values['Data_Item'].isin([
        'Choice beef byproduct value',
        'Choice beef gross farm value',
        'All fresh beef retail value'
    ])
]

In [307]:
fig = px.line(
    beef_values,
    x='Year',
    y='Value',
    color='SCM_Stage',
    title='Beef Price Transmission Across the Supply Chain'
)

fig.show()

###### This chart compares beef prices across farm, wholesale, and retail supply-chain stages. Retail prices rise sharply over time and remain consistently higher than wholesale and farm values. The graph also shows significant price volatility and strong increases after 2000, particularly in the retail stage.

##### Chicken Values

In [308]:
chicken_values = clean_df[
    (clean_df['Meat_Type'] == 'Chicken') &
    (clean_df['Metric_Type'] == 'Value')
]

In [309]:
fig = px.line(
    chicken_values,
    x='Year',
    y='Value',
    color='SCM_Stage',
    title='Chicken Price Transmission Across the Supply Chain',
    color_discrete_map={
        'Farm': 'blue',
        'Wholesale': 'red',
        'Retail': 'green'
    }
)
fig.show()

###### This chart compares chicken prices across wholesale and retail supply-chain stages. Farm-level prices are not included in the graph, likely because consistent farm-stage chicken price data was unavailable or measured differently from wholesale and retail values. Retail prices remain consistently above wholesale prices throughout the period, while both stages show gradual long-term growth with moderate fluctuations and noticeable price increases after 2020.

In [310]:
spread_df.head(5)

,Year,Month,Month-number,Data_Item,Value,Units,SCM_Stage,Meat_Type,Metric_Type,Spread_Type
5,1970,January,1,Pork farm to retail price spread,33.4,Cents per pound of retail equivalent,Spread,Pork,Spread,Farm to Retail
6,1970,January,1,Pork farm to wholesale price spread,23.2,Cents per pound of retail equivalent,Spread,Pork,Spread,Farm to Wholesale
7,1970,January,1,Pork Wholesale to retail price spread,10.2,Cents per pound of retail equivalent,Spread,Pork,Spread,Wholesale to Retail
13,1970,January,1,Choice beef farm to retail price spread,37.1,Cents per pound of retail equivalent,Spread,Beef,Spread,Farm to Retail
14,1970,January,1,Choice beef farm to wholesale price spread,14.5,Cents per pound of retail equivalent,Spread,Beef,Spread,Farm to Wholesale


##### Spread Analysis

In [311]:
clean_df.head(5)

,Year,Month,Month-number,Data_Item,Value,Units,SCM_Stage,Meat_Type,Metric_Type,Spread_Type
0,1970,January,1,Pork byproduct value,4.3,Cents per pound of retail equivalent,Farm,Pork,Byproduct,Other
1,1970,January,1,Pork gross farm value,52.3,Cents per pound of retail equivalent,Farm,Pork,Value,Other
2,1970,January,1,Pork net farm value,48.0,Cents per pound of retail equivalent,Farm,Pork,Value,Other
3,1970,January,1,Pork wholesale value,71.2,Cents per pound of retail equivalent,Wholesale,Pork,Value,Other
4,1970,January,1,Pork retail value,81.4,Cents per pound of retail equivalent,Retail,Pork,Value,Other


In [312]:
fig = px.line(
    spread_df,
    x='Year',
    y='Value',
    color='Spread_Type',
    facet_row='Meat_Type',
    title='Supply Chain Price Spreads by Meat Type',
    color_discrete_map={
        'Farm to Retail': 'green',
        'Wholesale to Retail': 'red',
        'Farm to Wholesale': 'blue'
    }
)
fig.show()

In [313]:
pork_spreads = spread_df[spread_df['Meat_Type'] == 'Pork']
fig = px.line(
    pork_spreads,
    x='Year',
    y='Value',
    color='Spread_Type',
    title='Pork Supply Chain Price Spreads',
    color_discrete_map={
        'Farm to Retail': 'green',
        'Wholesale to Retail': 'red',
        'Farm to Wholesale': 'blue'
    }
)
fig.show()

In [314]:
beef_spreads = spread_df[spread_df['Meat_Type'] == 'Beef']
fig = px.line(
    beef_spreads,
    x='Year',
    y='Value',
    color='Spread_Type',
    title='Beef Supply Chain Price Spreads',
    color_discrete_map={
        'Farm to Retail': 'green',
        'Wholesale to Retail': 'red',
        'Farm to Wholesale': 'blue'
    }
)
fig.show()

In [315]:
chicken_spreads = spread_df[spread_df['Meat_Type'] == 'Chicken']
fig = px.line(
    chicken_spreads,
    x='Year',
    y='Value',
    color='Spread_Type',
    title='Chicken Supply Chain Price Spreads',
    color_discrete_map={
        'Farm to Retail': 'green',
        'Wholesale to Retail': 'red',
        'Farm to Wholesale': 'blue'
    }
)
fig.show()

#####   Correlation & Distribution Analysis

In [316]:
# Create a scatter plot comparing the wholesale and retail prices for pork, with a trendline
wide = clean_df.pivot_table(
    index='Year',
    columns='Data_Item',
    values='Value',
    aggfunc='mean'
).reset_index()

In [317]:
# Create a scatter plot comparing the wholesale and retail prices for pork, with a trendline
fig = px.scatter(
    wide,
    x='Pork net farm value',
    y='Pork retail value',
    trendline='ols',
    title='Pork Farm vs Retail Prices'
)

fig.show()

In [318]:
# Create a scatter plot comparing the wholesale and retail prices for beef, with a trendline    
fig = px.scatter(
    wide,
    x='Choice beef net farm value',
    y='Choice beef retail value',
    trendline='ols',
    title='Beef Farm vs Retail Prices'
)

fig.show()

In [319]:
# Create a scatter plot comparing the wholesale and retail prices for chicken, with a trendline
fig = px.scatter(
    wide,
    x='Wholesale broiler composite',
    y='Retail broiler composite',
    trendline='ols',
    title='Chicken Wholesale vs Retail Prices'
)

fig.show()

In [320]:
# Calculate the correlation of pork between farm, wholesale, and retail prices for pork
wide[
    [
        'Pork net farm value',
        'Pork wholesale value',
        'Pork retail value'
    ]
].corr()

Data_Item,Pork net farm value,Pork wholesale value,Pork retail value
Data_Item,,,
Pork net farm value,1.000000,0.857910,0.659074
Pork wholesale value,0.857910,1.000000,0.910735
Pork retail value,0.659074,0.910735,1.000000


In [321]:
# Calculate the correlation of beef between farm, wholesale, and retail prices for beef
wide[
    [
        'Choice beef net farm value',
        'Choice beef wholesale value',
        'Choice beef retail value'
    ]
].corr()

Data_Item,Choice beef net farm value,Choice beef wholesale value,Choice beef retail value
Data_Item,,,
Choice beef net farm value,1.000000,0.978588,0.965300
Choice beef wholesale value,0.978588,1.000000,0.991586
Choice beef retail value,0.965300,0.991586,1.000000


In [322]:
# Calculate the correlation of chicken between wholesale and retail prices for chicken
wide[
    [
        'Wholesale broiler composite',
        'Retail broiler composite'
    ]
].corr()

Data_Item,Wholesale broiler composite,Retail broiler composite
Data_Item,,
Wholesale broiler composite,1.000000,0.917149
Retail broiler composite,0.917149,1.000000


In [323]:
# Create a histogram of the distribution of the pork farm to retail price spread
fig = px.histogram(
    wide,
    x='Pork farm to retail price spread',
    title='Distribution of Pork Farm-to-Retail Spread'
)

fig.show()

In [324]:
# Scatter plot: Pork wholesale value vs Pork retail value
pork = df[df['Meat_Type'] == 'Pork']
pork_wide = pork.pivot_table(index=['Year', 'Month-number'], 
                              columns='Data_Item', 
                              values='Value').reset_index()

fig = px.scatter(
    pork_wide,
    x='Pork wholesale value',
    y='Pork retail value',
    color='Year',
    title='Pork Wholesale vs Retail Value (1970–2025)',
    labels={
        'Pork wholesale value': 'Wholesale Value (cents/lb retail equiv.)',
        'Pork retail value': 'Retail Value (cents/lb)'
    }
)

fig.show()